## **Problem Set 3**. Text I/II: classical representations, classification & embeddings
### *Goal: move from sparse count vectors to dense distributional semantics.*

In [6]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import classification_report
import spacy
from sklearn.pipeline import Pipeline
import gensim.downloader as api
from gensim.models import Word2Vec
import ssl

##### 1. **[Theory]** Derive the TF-IDF weight $w_{t,d} = \text{tf}_{t,d} \cdot \log \frac{N}{\text{df}_t}$. Give an information-theoretic reading of the $\log(N/\text{df}_t)$ term and explain why raw term frequency alone is a poor weighting.

TF-IDF (Term Frequency-Inverse Document Frequency)

1. **Term Frequency ($\text{tf}$)**

Term frequency measures how often a term appears within a document:

$$\text{tf}(t, d) = \frac{f_{t,d}}{\sum_{t' \in d} f_{t',d}}$$

Where:
* $f_{t,d}$ - the number of times term $t$ occurs in document $d$.
* The denominator is the total number of terms in document $d$.

2. **Why raw term frequency alone is a poor weighting?**

- Longer documents naturally repeat words more often, because they have more text, not because the topic is more important. For example, common helper words like **"the"**, **"a"**, or **"and"** will appear hundreds of times in a long text simply due to its size, completely unrelated to what the document is actually about.
- Finding a keyword 50 times doesn't make a document 50 times more useful than finding it just once. After the first few mentions, extra repeats don't add any real value.

3. **Inverse Document Frequency ($\text{idf}$)**

To give higher weight to terms that appear in fewer documents and lower weight to frequent ones, we use Inverse Document Frequency. 

The weight of the term is inversely proportional to the number of documents in which it is presented:
$$w_t \propto \frac{1}{\text{df}_t}$$

Consider a corpus of $N$ documents, where term $t$ appears in $\text{df}_t$ documents (Document Frequency). If we pick a random document from the corpus, the probability $P_t$ that it contains term $t$ is:

$$P_t = \frac{\text{df}_t}{N}$$

Using Shannon's information theory, the self-information of observing term $t$ is:

$$I(t) = -\log P_t = -\log \left( \frac{\text{df}_t}{N} \right) = \log \left( \frac{N}{\text{df}_t} \right) = idf(t)$$

* *High $\text{df}_t$ (popular/common words like "the", "and", "a"):* $P_t \approx 1$, which means $\log(N/\text{df}_t) \to 0$. These terms contribute very little information.
* *Low $\text{df}_t$ (Rare, specific words):* $P_t \ll 1$, meaning $\log(N/\text{df}_t)$ is high. These terms help in discriminating, distinguishing, and ranking documents, contributing more information to the corpus.

4. **TF-IDF Weight**
Combining all together:

$$w_{t,d} = \text{tf}_{t,d} \cdot \log \left( \frac{N}{\text{df}_t} \right)$$

##### 2. **[Theory]** *(Multinomial Naïve Bayes.)* Derive the MAP parameter estimates with Laplace smoothing, $\hat{\theta}_{t|c} = \frac{n_{t,c} + \alpha}{\sum_{t'} n_{t',c} + \alpha |V|}$. Show that the resulting decision rule is *linear* in the word-count vector, and compare its form to logistic regression.

**1. Derivation of the MAP Parameter Estimates** \
**Likelihood Function** \
In Multinomial Naïve Bayes, a document $d$ of length $L$ is represented as a word-count vector $x = (x_1, \dots, x_{\vert{}V\vert{}}) \in \mathbb{N}^{\vert{}V\vert{}}$. We model the joint generative process of a document and its class $c \in \mathcal{C}$ as:
   1. Draw the class $c \sim \text{Categorical}(\pi)$, where $\pi_c = P(c)$, $\pi_c \ge 0$, and $\sum_{c \in \mathcal{C}} \pi_c = 1$. \ 
   2. Draw the $L$ tokens independently from a class-specific categorical distribution parameterized by $\theta_c = (\theta_{1\vert{}c}, \dots, \theta_{\vert{}V\vert{}\vert{}c})$, where $\theta_{t\vert{}c} \ge 0$ and $\sum_{t \in V} \theta_{t\vert{}c} = 1$.   

The conditional probability of observing a document vector $x$ given class $c$ is:
   $$P(x \mid c) = \frac{L!}{\prod_{t \in V} x_t!} \prod_{t \in V} \theta_{t\vert{}c}^{x_t}$$

Suppose we have a training dataset $\mathcal{D} = \{(x^{(i)}, c_i)\}_{i=1}^N$ of $N$ documents. Let:   
- $N_c = \sum_{i=1}^N \mathbb{I}[c_i = c]$ be the number of documents in class $c$ (so $\sum_{c \in \mathcal{C}} N_c = N$),  
- $n_{t,c} = \sum_{i: c_i = c} x_t^{(i)}$ be the total count of token $t$ across all training documents belonging to class $c$. 

The parameters decouple across classes and between $\pi$ and $\{\theta_c\}_{c \in \mathcal{C}}$. Dropping constants independent of the parameters, the full data likelihood is:
$$P(\mathcal{D} \mid \pi, \theta) \propto \left( \prod_{c \in \mathcal{C}} \pi_c^{N_c} \right) \prod_{c \in \mathcal{C}} \prod_{t \in V} \theta_{t\vert{}c}^{n_{t,c}}$$

**Prior Distributions (Dirichlet)** \
To perform Maximum A Posteriori (MAP) estimation and prevent zero probabilities for unseen words, we place a conjugate Dirichlet prior on $\theta_c$ for each class. Using symmetric hyperparameters $\beta_t = 1 + \alpha$ (where $\alpha > 0$, and $\alpha = 1$ corresponds to standard Laplace smoothing):
   $$P(\theta_c) \propto \prod_{t \in V} \theta_{t\vert{}c}^{\beta_t - 1} = \prod_{t \in V} \theta_{t\vert{}c}^{\alpha}$$
   
For the class prior $\pi$, we use either a flat (uniform) prior $P(\pi) \propto 1$ (which yields the MLE) or a symmetric Dirichlet prior $P(\pi) \propto \prod_{c \in \mathcal{C}} \pi_c^{\gamma}$ with $\gamma \ge 0$. 

**Posterior and Lagrangian Optimization for $\theta_c$** \
By Bayes' theorem, the posterior for $\theta_c$ is proportional to the product of its likelihood and prior:
$$P(\theta_c \mid \mathcal{D}) \propto \prod_{t \in V} \theta_{t\vert{}c}^{n_{t,c} + \alpha}$$

To find the MAP estimate $\hat{\theta}_c$, we maximize the log-posterior subject to the probability simplex constraint $\sum_{t \in V} \theta_{t\vert{}c} = 1$:
$$\mathcal{L}(\theta_c, \lambda) = \sum_{t \in V} (n_{t,c} + \alpha) \log \theta_{t\vert{}c} - \lambda \left( \sum_{t \in V} \theta_{t\vert{}c} - 1 \right)

$$Taking the partial derivative with respect to $\theta_{t\vert{}c}$ and setting it to zero:
$$\frac{\partial \mathcal{L}}{\partial \theta_{t\vert{}c}} = \frac{n_{t,c} + \alpha}{\theta_{t\vert{}c}} - \lambda = 0 \implies \theta_{t\vert{}c} = \frac{n_{t,c} + \alpha}{\lambda}$$

Substituting this into the normalization constraint $\sum_{t' \in V} \theta_{t'\vert{}c} = 1$ gives the Lagrange multiplier $\lambda$:
$$\sum_{t' \in V} \frac{n_{t',c} + \alpha}{\lambda} = 1 \implies \lambda = \sum_{t' \in V} (n_{t',c} + \alpha) = \sum_{t' \in V} n_{t',c} + \alpha \vert{}V\vert{}$$

Substituting $\lambda$ back yields the MAP parameter estimate:
$$\hat{\theta}_{t\vert{}c} = \frac{n_{t,c} + \alpha}{\sum_{t' \in V} n_{t',c} + \alpha \vert{}V\vert{}}$$

*Why this is a unique maximum:* Because $n_{t,c} + \alpha > 0$ for all $t \in V$ and $\log(\cdot)$ is strictly concave, the objective $\sum_{t \in V} (n_{t,c} + \alpha) \log \theta_{t\vert{}c}$ is strictly concave on the probability simplex. Furthermore, the non-negativity constraints $\theta_{t\vert{}c} \ge 0$ never bind because $\log \theta_{t\vert{}c} \to -\infty$ as $\theta_{t\vert{}c} \to 0^+$. Thus, the stationary point found via the Lagrangian is the unique global maximizer.

**Estimation of the Class Prior $\hat{\pi}_c$**
Applying the exact same Lagrangian derivation to the log-posterior of $\pi$, $\sum_{c \in \mathcal{C}} (N_c + \gamma) \log \pi_c - \mu \left(\sum_{c \in \mathcal{C}} \pi_c - 1\right)$, yields:$$\hat{\pi}_c = \frac{N_c + \gamma}{N + \gamma \vert{}\mathcal{C}\vert{}}$$

Under a flat prior ($\gamma = 0$), this simplifies directly to the empirical class frequency:
   $$\hat{\pi}_c = \frac{N_c}{N}$$


**2. Linearity of the Decision Rule**
To classify a new document $x$, we apply Bayes' rule to find the class $c$ that maximizes the posterior probability $P(c \mid x)$:
$$\hat{c} = \arg\max_{c} P(c \mid x) = \arg\max_{c} \frac{P(x \mid c) P(c)}{P(x)}$$

Since $P(x)$ is constant across all classes and the logarithm is strictly monotonically increasing, maximizing $P(c \mid x)$ is equivalent to maximizing $\log P(x \mid c) + \log P(c)$:
$$\hat{c} = \arg\max_{c} \left[ \log \pi_c + \log \left( \frac{L!}{\prod_{t \in V} x_t!} \prod_{t \in V} \hat{\theta}_{t\vert{}c}^{x_t} \right) \right]$$

Expanding the logarithm and dropping the multinomial term $\log\left(\frac{L!}{\prod_{t \in V} x_t!}\right)$ (since it does not depend on $c$):
$$\hat{c} = \arg\max_{c} \left[ \log \pi_c + \sum_{t \in V} x_t \log \hat{\theta}_{t\vert{}c} \right]$$

Let $w_c = \left(\log \hat{\theta}_{1\vert{}c}, \dots, \log \hat{\theta}_{\vert{}V\vert{}\vert{}c}\right)^\top \in \mathbb{R}^{\vert{}V\vert{}}$ be the weight vector and $b_c = \log \pi_c$ be the bias for class $c$. The decision function becomes:
$$\hat{c} = \arg\max_{c} \left( \langle w_c, x \rangle + b_c \right)$$

For the binary classification case ($c \in \{+, -\}$), the log-odds decision boundary is given by:
$$\log \frac{P(+ \mid x)}{P(- \mid x)} = \log \frac{\pi_+}{\pi_-} + \sum_{t \in V} x_t \log \frac{\hat{\theta}_{t\vert{}+}}{\hat{\theta}_{t\vert{}-}} = \langle w, x \rangle + b > 0$$

where $w_t = \log \hat{\theta}_{t\vert{}+} - \log \hat{\theta}_{t\vert{}-}$ and $b = \log \pi_+ - \log \pi_-$. Because the decision score is an affine (linear) function of the word-count vector $x$, the classifier is a linear classifier.   

**3. Comparison with Logistic Regression**

##### 3. **[Theory]** *(Skip-gram with negative sampling.)* Write the SGNS objective $\log \sigma(v_c^\top v_w) + \sum_{i=1}^k \mathbb{E}_{w_i \sim P_n} \log \sigma(-v_{w_i}^\top v_w)$. Derive the gradient w.r.t. a centre-word vector $v_w$, and explain informally why SGNS approximates a shifted PMI factorisation.

##### 4. **[Python]** Build a TF-IDF pipeline ($\text{spaCy/NLTK preprocessing} \to \text{TfidfVectorizer}$) and train both multinomial Naïve Bayes and logistic regression on a text-classification dataset (e.g. 20 Newsgroups subset). Report precision/recall/F1 and list the most informative features per class.

- [The 20 Newsgroups data set](http://qwone.com/~jason/20Newsgroups/)

The 20 Newsgroups data set is a collection of approximately $20 000$ newsgroup documents, partitioned (nearly) evenly across 20 different newsgroups. Some of the newsgroups are very closely related to each other (e.g. `comp.sys.ibm.pc.hardware / comp.sys.mac.hardware`), while others are highly unrelated (e.g `misc.forsale / soc.religion.christian`).

Here is a list of the 20 newsgroups, partitioned (more or less) according to subject matter:

$$
\begin{array}{|l|l|l|}
\hline
\begin{array}{l} 
\text{comp.graphics} \\ 
\text{comp.os.ms-windows.misc} \\ 
\text{comp.sys.ibm.pc.hardware} \\ 
\text{comp.sys.mac.hardware} \\ 
\text{comp.windows.x} 
\end{array} & 
\begin{array}{l} 
\text{rec.autos} \\ 
\text{rec.motorcycles} \\ 
\text{rec.sport.baseball} \\ 
\text{rec.sport.hockey} 
\end{array} & 
\begin{array}{l} 
\text{sci.crypt} \\ 
\text{sci.electronics} \\ 
\text{sci.med} \\ 
\text{sci.space} 
\end{array} \\
\hline
\begin{array}{l} 
\text{misc.forsale} 
\end{array} & 
\begin{array}{l} 
\text{talk.politics.misc} \\ 
\text{talk.politics.guns} \\ 
\text{talk.politics.mideast} 
\end{array} & 
\begin{array}{l} 
\text{talk.religion.misc} \\ 
\text{alt.atheism} \\ 
\text{soc.religion.christian} 
\end{array} \\
\hline
\end{array}
$$


In [2]:
cats = ['alt.atheism', 'sci.space', 'comp.windows.x', 'rec.sport.baseball']
newsgroups_train = fetch_20newsgroups(subset='train', categories=cats)
newsgroups_test = fetch_20newsgroups(subset='test', categories=cats)

# python -m spacy download en_core_web_sm
def spacy_preprocessor(texts):
    nlp = spacy.load("en_core_web_sm")
    preprocessed_texts = []
    for doc in nlp.pipe(texts, disable=["parser", "ner"]):
        tokens = [token.lemma_.lower() for token in doc if not token.is_stop and not token.is_punct]
        preprocessed_texts.append(" ".join(tokens))
    return preprocessed_texts

In [17]:
def train_tfidf_models(X_train, y_train, X_test, y_test, target_names):
    X_train_clean = spacy_preprocessor(X_train)
    X_test_clean = spacy_preprocessor(X_test)

    vectorizer = TfidfVectorizer(max_df=0.9, min_df=2)
    X_train_tfidf = vectorizer.fit_transform(X_train_clean)
    X_test_tfidf = vectorizer.transform(X_test_clean)

    models = {
        'Multinomial Naive Bayes': MultinomialNB(),
        'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0)
    }

    results = {}
    feature_names = np.array(vectorizer.get_feature_names_out())

    for name, classifier in models.items():
        classifier.fit(X_train_tfidf, y_train)
        y_pred = classifier.predict(X_test_tfidf)

        report = classification_report(
            y_test,
            y_pred,
            target_names=target_names,
            output_dict=True,
            zero_division=0
        )

        if hasattr(classifier, 'coef_'):
            weights = classifier.coef_
        else:
            weights = classifier.feature_log_prob_

        top_features = {}

        for i, class_name in enumerate(target_names):
            top_indices = weights[i].argsort()[-5:][::-1]
            top_features[class_name] = feature_names[top_indices].tolist()

        results[name] = {
            'model': classifier,
            'vectorizer': vectorizer,
            'predictions': y_pred,
            'report': report,
            'top_features': top_features
        }

    return results

In [18]:
results_tfidf = train_tfidf_models(
    newsgroups_train.data,
    newsgroups_train.target,
    newsgroups_test.data,
    newsgroups_test.target,
    newsgroups_train.target_names
)

In [19]:
for model, result in results_tfidf.items():
    print(f'\n=== {model} ===')
    print(classification_report(
        newsgroups_test.target,
        result['predictions'],
        target_names=newsgroups_train.target_names,
        zero_division=0
    ))


=== Multinomial Naive Bayes ===
                    precision    recall  f1-score   support

       alt.atheism       0.99      0.95      0.97       319
    comp.windows.x       0.98      0.97      0.97       395
rec.sport.baseball       0.98      0.99      0.98       397
         sci.space       0.95      0.97      0.96       394

          accuracy                           0.97      1505
         macro avg       0.97      0.97      0.97      1505
      weighted avg       0.97      0.97      0.97      1505


=== Logistic Regression ===
                    precision    recall  f1-score   support

       alt.atheism       0.99      0.92      0.95       319
    comp.windows.x       0.96      0.98      0.97       395
rec.sport.baseball       0.96      0.99      0.98       397
         sci.space       0.96      0.96      0.96       394

          accuracy                           0.97      1505
         macro avg       0.97      0.96      0.97      1505
      weighted avg       0.97    

In [20]:
for model, result in results_tfidf.items():
    print(f'\n=== {model} ===')
    for class_name, features in result['top_features'].items():
        print(f'{class_name}: {", ".join(features)}')


=== Multinomial Naive Bayes ===
alt.atheism: edu, god, keith, com, write
comp.windows.x: window, com, edu, motif, server
rec.sport.baseball: edu, game, team, year, baseball
sci.space: space, edu, nasa, henry, access

=== Logistic Regression ===
alt.atheism: god, keith, people, atheist, atheism
comp.windows.x: window, motif, widget, server, mit
rec.sport.baseball: baseball, game, team, player, year
sci.space: space, nasa, orbit, moon, launch


##### 5. **[Python]** Train word2vec with `gensim` on the same corpus and separately load pretrained GloVe. For both, return the nearest neighbours of five query words and evaluate on an analogy set (e.g. $\text{king} - \text{man} + \text{woman}$)

In [ ]:
def train_word2vec(X_train_raw, vector_size=100, window=5, min_count=2, epochs=20):
    X_train_clean = spacy_preprocessor(X_train_raw)
    corpus = [text.split() for text in X_train_clean if text.strip()]

    return Word2Vec(
        sentences=corpus,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=1,
        sg=1,
        epochs=epochs,
        seed=42)


def word2vec_glove_pipeline(X_train_raw, model_name='glove-wiki-gigaword-100'):
    w2v = train_word2vec(X_train_raw)
    glove = api.load(model_name)

    models = {'Word2Vec': w2v.wv, 'GloVe': glove}

    # Nearest neighbours
    query_words = ['space', 'god', 'window', 'baseball', 'computer']
    neighbors = {}
    for name, model in models.items():
        neighbors[name] = {
            word: model.most_similar(word, topn=5) if word in model else None
            for word in query_words
        }

    # Word analogies
    analogy_set = [
        ('king', 'man', 'woman', 'queen'),
        ('brother', 'man', 'woman', 'sister'),
        ('father', 'man', 'woman', 'mother'),
        ('paris', 'france', 'germany', 'berlin'),
        ('uncle', 'man', 'woman', 'aunt')
    ]

    analogies = {}
    for name, model in models.items():
        details = []
        correct = 0
        evaluated = 0
        skipped = 0
        for a, b, c, expected in analogy_set:
            if not all(word in model for word in [a, b, c, expected]):
                skipped += 1
                continue

            predicted = model.most_similar(positive=[a, c], negative=[b], topn=1)[0][0]
            is_correct = predicted == expected
            correct += is_correct
            evaluated += 1

            details.append({
                'analogy': f'{a} - {b} + {c}',
                'expected': expected,
                'predicted': predicted,
                'correct': is_correct
            })

        analogies[name] = {
            'accuracy': correct / evaluated if evaluated else None,
            'correct': correct,
            'evaluated': evaluated,
            'skipped': skipped,
            'details': details
        }

    return {
        'word2vec_model': w2v,
        'glove_model': glove,
        'neighbors': neighbors,
        'analogies': analogies
    }

In [10]:
results_word2vec_glove = word2vec_glove_pipeline(newsgroups_train.data)

[==================================================] 100.0% 128.1/128.1MB downloaded


In [11]:
for model, words in results_word2vec_glove['neighbors'].items():
    print(f'\n== Neighbors for {model} ==')

    for word, neighbors in words.items():
        print(f'{word}: {neighbors}')


== Neighbors for Word2Vec ==
space: [('station', 0.693341851234436), ('digest', 0.6041110754013062), ('shuttle', 0.5904091596603394), ('groups', 0.5821332931518555), ('larrison', 0.5775603652000427)]
god: [('dice', 0.6722433567047119), ('kierkegaard', 0.6608384847640991), ('hinge', 0.6598546504974365), ('nonexistence', 0.6542344093322754), ('ital', 0.6538872718811035)]
window: [('decoration', 0.7350329160690308), ('resize', 0.7176738381385803), ('revert', 0.6864197850227356), ('xquerytree', 0.6770200729370117), ('tvtwm', 0.6652793288230896)]
baseball: [('players', 0.6767725348472595), ('rivers', 0.6580016016960144), ('football', 0.6579993367195129), ('broadcasters', 0.6506617069244385), ('multipurpose', 0.6502743363380432)]
computer: [('concurrent', 0.6991081237792969), ('junior', 0.6779113411903381), ('vpi&su', 0.6613692045211792), ('science', 0.6532675623893738), ('sutherland', 0.6491221785545349)]

== Neighbors for GloVe ==
space: [('nasa', 0.7037306427955627), ('spaces', 0.6882482

In [12]:
for model, result in results_word2vec_glove['analogies'].items():
    print(f'\n== Evaluation on an analogy set for {model} ==')
    print(f"Accuracy: {result['accuracy']}")
    print(f"Evaluated: {result['evaluated']}")
    print(f"Skipped: {result['skipped']}")

    for item in result['details']:
        print(item)


== Evaluation on an analogy set for Word2Vec ==
Accuracy: 0.0
Evaluated: 4
Skipped: 1
{'analogy': 'king - man + woman', 'expected': 'queen', 'predicted': 'dmn@kepler.unh.edu', 'correct': False}
{'analogy': 'brother - man + woman', 'expected': 'sister', 'predicted': 'sod', 'correct': False}
{'analogy': 'father - man + woman', 'expected': 'mother', 'predicted': 'miscarriage', 'correct': False}
{'analogy': 'paris - france + germany', 'expected': 'berlin', 'predicted': 'brother', 'correct': False}

== Evaluation on an analogy set for GloVe ==
Accuracy: 0.8
Evaluated: 5
Skipped: 0
{'analogy': 'king - man + woman', 'expected': 'queen', 'predicted': 'queen', 'correct': True}
{'analogy': 'brother - man + woman', 'expected': 'sister', 'predicted': 'daughter', 'correct': False}
{'analogy': 'father - man + woman', 'expected': 'mother', 'predicted': 'mother', 'correct': True}
{'analogy': 'paris - france + germany', 'expected': 'berlin', 'predicted': 'berlin', 'correct': True}
{'analogy': 'uncle -

##### 6. **[Analysis]** Probe the embeddings for social bias (e.g. gendered analogies or association tests). Report what you find in one paragraph and comment on the risk of propagating such bias downstream.